In [1]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("emirhanai/human-action-detection-artificial-intelligence")

print("Path to dataset files:", path)

BackendError: POST failed with: {"errors":["New Datasets cannot be attached in non-interactive sessions. Found no versions attached for Dataset [emirhanai/human-action-detection-artificial-intelligence]."],"error":{"code":9},"wasSuccessful":false}

In [ ]:
path

In [ ]:
from transformers import MobileViTImageProcessor, MobileViTForImageClassification,PretrainedConfig,PreTrainedModel,BitsAndBytesConfig
from peft import LoraConfig, get_peft_model
from PIL import Image
import requests

image = Image.open(path+"/emirhan_human_dataset/datasets/human_data/train_data/calling/images_083.jpg")





In [ ]:
classes=[]
#read classes
with open(path+"/emirhan_human_dataset/datasets/data.txt","r") as f:
    for line in f:
        classes.append(line.strip())

In [ ]:
!pip install -U torchao

In [ ]:
import torch

In [ ]:
class ModeruConfig(PretrainedConfig):
    model_type = "Moderu"

    def __init__(
        self,
        n_labels,n_dense,cudas,
        **kwargs
    ):
        super().__init__(**kwargs)
        self.n_labels=n_labels
        self.n_dense=n_dense
        self.cuda=cudas


In [ ]:
class Moderu(PreTrainedModel):
    config_class = PretrainedConfig
    def __init__(self,config):
        super().__init__(config)
        self.fe=MobileViTImageProcessor.from_pretrained("apple/mobilevit-small")
        self.bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
)

        self.vit=MobileViTForImageClassification.from_pretrained("apple/mobilevit-small")
        self.loraconfig=LoraConfig(

    r=6,
    lora_alpha=6,
    target_modules=["query","key","value"],
    lora_dropout=0.1,
    bias="none",
    modules_to_save=["head"],

        )

        self.vit=get_peft_model(self.vit,self.loraconfig)
        self.classifier1=torch.nn.Sequential(torch.nn.Conv2d(160,640,kernel_size=(2,2)),
                              torch.nn.BatchNorm2d(640),
                              torch.nn.AdaptiveAvgPool2d(2),
                              torch.nn.Flatten(),
                              torch.nn.Linear(2560,600),
                              torch.nn.LayerNorm(600)



                            )
        #2560
        intermediate=[]
        for _ in range(config.n_dense):
            intermediate+=[torch.nn.GELU(),torch.nn.Linear(600,600) ]
        intermediate+=[torch.nn.Linear(600,config.n_labels)]
        self.classifier2=torch.nn.Sequential(*intermediate)
        self.cudat=config.cuda
        if self.cudat:
          self.to("cuda")



    def forward(self,x):
        x=self.fe(images=x,return_tensors="pt").to("cuda") if self.cudat else self.fe(images=x,return_tensors="pt")

        x=self.vit(**x,output_hidden_states=True).hidden_states[-1]

        x=self.classifier1(x)
        x=self.classifier2(x)

        return x




In [ ]:
inputs = feature_extractor(images=image, return_tensors="pt")

outputs = model(**inputs, output_hidden_states=True)

In [ ]:
loraconfig=LoraConfig(

    r=6,
    lora_alpha=6,
    target_modules=["query","key","value"],
    lora_dropout=0.1,
    bias="none",
    modules_to_save=["head"],

        )
vit=get_peft_model(model,loraconfig)

In [ ]:
#get number trainable parameters
for name, param in vit.named_parameters():
    if param.requires_grad:
        print(name, param.numel())

In [ ]:
import __main__

__main__.__file__ = "notebook.py"

In [ ]:
with open("notebook.py", "w") as f:
    f.write("# dummy notebook file")

In [ ]:
moderu=Moderu(ModeruConfig(len(classes),1,True))

In [ ]:
pred=moderu([image,image])
loss(pred,torch.tensor([[1,0],[0,1]],dtype=torch.float).cuda())


In [ ]:
import random
import os

In [ ]:
dataset_x=[]
dataset_y=[]

In [ ]:
#feed dataset
n_samples=10000
for i in range(n_samples):
  random_class=random.randint(0,len(classes)-1)
  #ls training data directory
  tr_d=os.listdir(path+"/emirhan_human_dataset/datasets/human_data/train_data/")
  tr_d=tr_d[random_class]
  tr_chosen=os.listdir(path+"/emirhan_human_dataset/datasets/human_data/train_data/"+tr_d)
  chosen=random.randint(0,len(tr_chosen)-1)
  img=Image.open(path+"/emirhan_human_dataset/datasets/human_data/train_data/"+tr_d+"/"+tr_chosen[chosen])
  dataset_x.append(img)
  dataset_y.append([1 if i==random_class else 0 for i in range(len(classes))])

  #choose random image




In [ ]:
#load the dataset but first shuffle it

cross_validation_idx=0

In [ ]:
from itertools import chain

In [ ]:
#optimizer=torch.optim.Adam(moderu.parameters(),lr=1e-4)
#optimizer.zero_grad()
#loss=torch.nn.CrossEntropyLoss()
cross_validation=False
k=1
training=[]
#eval=[]
t_steps=1
data_training_x=[]
data_training_y=[]
validation_x=[]
validation_y=[]
batch_size=16
if cross_validation:
  #folds
  data_training_x=[dataset_x[i:i+k] for i in range(0,len(dataset_x),k)]
  data_training_y=[dataset_y[i:i+k] for i in range(0,len(dataset_y),k)]
  validation_x=data_training_x[cross_validation_idx]
  validation_y=data_training_y[cross_validation_idx]
  data_training_x.pop(cross_validation_idx)
  data_training_y.pop(cross_validation_idx)
  data_training_x=list(chain(*data_training_x))
  data_training_y=list(chain(*data_training_y))
  cross_validation_idx+=1
  if cross_validation_idx==len(data_training_x):
    cross_validation_idx=0
else:
  data_training_x=dataset_x[:int(len(dataset_x)*0.8)]
  data_training_y=dataset_y[:int(len(dataset_y)*0.8)]
  validation_x=dataset_x[int(len(dataset_x)*0.8):]
  validation_y=dataset_y[int(len(dataset_y)*0.8):]





for i in range(0,len(data_training_x),batch_size):
  batch_x=data_training_x[i:i+batch_size]
  batch_y=data_training_y[i:i+batch_size]
  prediction=moderu(batch_x)
  l=loss(prediction,torch.tensor(batch_y,dtype=torch.float).to("cuda")) if moderu.cudat else  loss(prediction,torch.tensor(batch_y,dtype=torch.float))
  l.backward()

  torch.nn.utils.clip_grad_norm_(moderu.parameters(), 1.0)

  optimizer.step()
  optimizer.zero_grad()
  training.append(l.item())
  print(i)

for i in range(0,len(validation_x),batch_size):
  batch_x=validation_x[i:i+batch_size]
  batch_y=validation_y[i:i+batch_size]
  with torch.no_grad():
    prediction=moderu(batch_x)
    l=loss(prediction,torch.tensor(batch_y,dtype=torch.float).to("cuda")) if moderu.cudat else  loss(prediction,torch.tensor(batch_y,dtype=torch.float))
  eval.append(l.item())
plt.plot(eval)








In [ ]:
import matplo

In [ ]:
outputs.hidden_states[-1].shape

In [ ]:
testlayer=

In [ ]:
testlayer(outputs.hidden_states[-1]).shape